In [2]:
import torch

## A bit about ChatGPT:
- ChatGPT = Probabalistic Model, might give different answers for the same prompt
- it's a language model, knows how words follow (you give the start of something, the AI predicts the next words)
- it's architecture comes from "attention is all you need" 2027 paper on the transformer model
- transformer model is the NN architecture that is doing a lot of heavy lifting in ChatGPt

## NanoGPT:
- character level language model (not word level just yet)
- trained on the shakespearean dataset for our tutorial with karpathy

## wyd:
1. write a transformer piece by piece
2. train on tiny skspr (br i n skspr)
3. will lead to infinite generation of sksprn


In [3]:
# reading tinyshakespeare:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of the dataset in chars:", len(text))
print()
print("first 500 characters roughly:")
print(text[:500])

length of the dataset in chars: 1115394

first 500 characters roughly:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [4]:
# getting all of the unique characters that occur in this text:

chars = sorted(list(set(text))) # chars that the model can see and emit
vocab_size = len(chars)
print('chars here:', ''.join(chars))
print('vocab size:', vocab_size)

chars here: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab size: 65


In [5]:
# tokenization of the input text:
# tokenize: raw text -> some some sequence of integers based on a vocabulary mapping
# here, we'll translate individual characters to numbers (ids) since we're doing a character level language model here

# just a simple lookup table here:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s] # takes a string, outputs a list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # takes a list of ints, outputs a string

print(encode("hiiii there"))
print(decode(encode("hiiii there")))

[46, 47, 47, 47, 47, 1, 58, 46, 43, 56, 43]
hiiii there


In [6]:
# converting to pytorch tensor array to treat tokenized chars as input:
data = torch.tensor(encode(text)) # massive sequence of integers, direct translation of the dataset of text as a long sequence
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [7]:
# splitting into training and validation set:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:] # to get a sense of overfitting

## Training Method:
- method 1: feeding the entire dataset all at once to the Neural Network ❌
- method 2: splitting the data into manageable chunks, and training the NN on multiple of these manageable chunks at the same time ✅🧠

## Training using Chunking:
- when splitting data into chunks (here, splitting data into chunks of n number of chars), the max length of a chunk is called the "block size"
- here, we use block size = 8
- done so that we can give the NN few pieces at a time rather than giving it everything (I would give the world to you ahh)

In [8]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

## Why block_size + 1?
- this leads to a character slice of 9 characters, then why we say block size 8?
- this is because when we look at how we'll train the NN:
    - for first character (id 18), how likely is the next one to be 47 optimization
    - for the first two chars taken at the same time, how likely is the next one 57
    - for the first three chars taken at the same time, how likely is the next one 58
    - and so on and so forth, forming 8 total examples
- This way, the NN is used to seeing examples that contain characters from size = 1 all the way to block size and all of the lengths in between (useful during inference, we can start with as little as 1 character to start)
- after we reach block size during inference, we start truncating (to be coming later)

This is a better visualization of what im tryna lay here:

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"block component number: {t+1}, when input is {context}, the target: {target}")

block component number: 1, when input is tensor([18]), the target: 47
block component number: 2, when input is tensor([18, 47]), the target: 56
block component number: 3, when input is tensor([18, 47, 56]), the target: 57
block component number: 4, when input is tensor([18, 47, 56, 57]), the target: 58
block component number: 5, when input is tensor([18, 47, 56, 57, 58]), the target: 1
block component number: 6, when input is tensor([18, 47, 56, 57, 58,  1]), the target: 15
block component number: 7, when input is tensor([18, 47, 56, 57, 58,  1, 15]), the target: 47
block component number: 8, when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target: 58


## Parallel Processing (multiple blocks at once in training)
- GPUs are very good at parallel processing, so what we boutta do is to independently train the NN on these different chunks simultaneously for efficiency
- parallel processing can be done by introducing the "batch dimension"
- batch dimension: how many independent sequences (blocks) will we process in parallel

In [23]:
torch.manual_seed(1337)
batch_size = 4 # 4 blocks trained in parallel
block_size = 8 # max context length for predictions

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == "train" else val_data # js a distinction which data we're going for 

    # picking the starting indexes of 4 random batches from the data
    # this results in a (4x1) tensor because the shape specified below is (batch_size, ), meaning (4,1)
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i : block_size + i] for i in ix]) # 4 blocks, each block having 4 character ids in them, stacked as rows in a tensor
    y = torch.stack([data[i + 1 : block_size + i + 1] for i in ix]) # same as x, j one ppos shifted over
    # both of these lead to (4x8) tensors

    # relation between x and y:
    # for index 0 -> i in x (i is range(8)), prediction should end up being y[i]

    return x, y

xb, yb = get_batch('train')
# xb's role: input
# yb's role: generate a loss function at the end of forward pass stage

print(f"Input size: {xb.shape}")
print(f"Input value: {xb}")
print(f"Output size: {yb.shape}")
print(f"Output value: {yb}")
print()

# relation of xb and yb (or just x and y), as showcased by the first row of x and y:
for i in range(block_size):
    example_input_row = xb[0]
    example_target_row = yb[0]

    print(f"when input is {example_input_row[:i+1]}, output should be: {example_target_row[i]}")

# note: each of the above 8 is an example, and there are 24 more examples (32 examples within these 4 batches themselves) to train the NN in one pass

Input size: torch.Size([4, 8])
Input value: tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Output size: torch.Size([4, 8])
Output value: tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])

when input is tensor([24]), output should be: 43
when input is tensor([24, 43]), output should be: 58
when input is tensor([24, 43, 58]), output should be: 5
when input is tensor([24, 43, 58,  5]), output should be: 57
when input is tensor([24, 43, 58,  5, 57]), output should be: 1
when input is tensor([24, 43, 58,  5, 57,  1]), output should be: 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]), output should be: 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]), output should be: 39
